# 8.3 章节实践

## 一、客观题

1. memcheck、racecheck、initcheck、synccheck 分别对应哪类故障？
2. 独立 ASC 与标准算子工程的 sanitizer 插桩位置分别是什么？
3. 故障程序返回非零是否等于实验失败？还必须检查什么？
4. `/dev/drv_debug` 不可读写时，msDebug 应标记 PASS、FAIL 还是 BLOCKED？
5. 判断：课程可以用 `sudo chmod` 临时开放 `/dev/drv_debug`。


## 二、简单编程题：memcheck

在独立 ASC 形态构建 memcheck 故障，捕获 mssanitizer 的实际退出码和完整日志。只有日志匹配 illegal read/write 或越界签名时，输出 `SIMPLE_STATUS=EXPECTED_DIAGNOSTIC`。


In [ ]:
%%bash
set -euo pipefail
MODE=memcheck
BUILD="build/chapter-simple-$MODE"
cmake -S src -B "$BUILD" -DLAB05_FAULT_MODE="$MODE" -DLAB05_ENABLE_SANITIZER=ON
cmake --build "$BUILD" -j
set +e
mssanitizer --tool="$MODE" "$BUILD/lab05_add_sanitizer" 2>&1 | tee "$BUILD/tool.log"
RC="${PIPESTATUS[0]}"
set -e
grep -Eiq 'illegal (read|write)|out[- ]of[- ]bounds' "$BUILD/tool.log"
echo "SIMPLE_STATUS=EXPECTED_DIAGNOSTIC mode=$MODE exit_code=$RC log=$BUILD/tool.log"


## 三、中等编程题：两种工程形态对照

从 racecheck、initcheck、synccheck 中任选一种，用独立 ASC 和标准算子工程各运行一次。两份日志必须匹配同一故障语义，同时说明为什么插桩 API 不同。建议通过环境变量 `DEBUG_MODE` 选择模式，默认使用 racecheck。


In [ ]:
import os

DEBUG_MODE = os.environ.get("DEBUG_MODE", "racecheck")
SIGNATURES = {
    "racecheck": r"RAW hazard|data race",
    "initcheck": r"uninitialized",
    "synccheck": r"unpaired[ _-]*set_flag|unpaired set_flag",
}
if DEBUG_MODE not in SIGNATURES:
    raise ValueError(f"DEBUG_MODE 必须是 {sorted(SIGNATURES)}")
print("选择模式：", DEBUG_MODE, "；预期签名：", SIGNATURES[DEBUG_MODE])


按 1.2 中的两条直接命令执行所选 `DEBUG_MODE`，分别保存 `standalone.log` 与 `operator.log`。检查项为：MODE 一致、两份日志均匹配 `SIGNATURES[DEBUG_MODE]`、退出码分别记录、诊断都能回到对应 Kernel 源码。不得只复制 memcheck 日志改名。


## 四、困难编程题：修复与统一报告

修复一个故障，重新运行 baseline 和对应工具，生成包含 `shape/mode/command/exit_code/signature/source/status` 的记录。msDebug 还要包含权限状态；权限不足时整体状态为 `BLOCKED`，不能写 PASS。

下面的函数只验证并写入学生提供的真实记录，不运行工具，也不预设结果。


In [ ]:
import json
from pathlib import Path


def write_debug_report(records, baseline_status, msdebug):
    required = {"shape", "mode", "command", "exit_code", "signature", "source", "status"}
    if not records:
        raise AssertionError("至少提供一条独立 ASC 和一条标准工程记录")
    shapes = {item.get("shape") for item in records}
    assert {"standalone", "operator"} <= shapes
    for item in records:
        missing = required - item.keys()
        if missing:
            raise AssertionError(f"记录缺少字段：{sorted(missing)}")
        assert item["status"] == "EXPECTED_DIAGNOSTIC"
    assert baseline_status == "PASS"
    assert msdebug.get("status") in {"PASS", "BLOCKED"}

    overall = "PASS" if msdebug["status"] == "PASS" else "BLOCKED"
    report = {
        "schema_version": 1,
        "records": records,
        "baseline_status": baseline_status,
        "msdebug": msdebug,
        "status": overall,
    }
    path = Path("work/01.03_chapter_test/debug_report.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"DEBUG_REPORT_STATUS={overall} report={path.resolve()}")
    return report

# 完成真实工具运行后再调用 write_debug_report(...)


## 完成标准

客观题正确；简单题得到 memcheck 预期诊断；中等题完成两种工程形态的同语义对照；困难题证明修复后的 baseline 正常并生成统一报告。msDebug 权限不足必须如实 `BLOCKED`，课程不得执行任何提权或系统权限修改。


In [ ]:
# 完成四类考核后按需执行；Notebook 不会自动展开答案。
!cat answer/08.03_answer.md
